# Clustering K-Means
## Dataset: Hotel Bookings

**K-Means** divide los datos en `k` grupos minimizando la distancia de cada punto
a su centroide (centro del grupo). A diferencia del clustering jerárquico, K-Means
requiere especificar el número de clusters de antemano, pero es mucho más rápido
en datasets grandes.

### Flujo de trabajo
```
mf.KMeans(path, num)        # 1. Cargar datos
    ↓ métodos EDA heredados  # 2. Limpiar y preparar
km.ajustar()                # 3. Ajustar el modelo
km.plot_*()                 # 4. Visualizar resultados
```

## 1. Importación de librerías

In [ ]:
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
import pckEDA as mf

%matplotlib inline
print('Clase KMeans disponible:', mf.KMeans)

## 2. Carga de datos

In [ ]:
km = mf.KMeans('../datasets/hotel_bookings.csv', 1, n_clusters=4)

km.mostrarTamaño()
km.muestraTiposDeDatos()

## 3. Preprocesamiento con métodos heredados del EDA

In [ ]:
km.codificarCategorica('hotel')
km.codificarCategorica('arrival_date_month')
km.codificarCategorica('assigned_room_type')
km.codificarCategorica('deposit_type', mapeo={'No Deposit': 0, 'Non Refund': 1, 'Refundable': 2})
km.codificarCategorica('customer_type')
km.codificarCategorica('reservation_status')
km.eliminarDuplicados()
km.eliminarNulos()
km.analisisNumerico()
print(f'Columnas para el modelo: {list(km.df.columns)}')

## 4. Selección del número óptimo de clusters

Antes de ajustar el modelo final, usamos el **método del codo** y el
**coeficiente de Silhouette** para elegir el mejor valor de `k`.

In [ ]:
# Necesitamos los datos escalados antes de graficar — ajustamos primero con k=4
km.ajustar()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plt.sca(axes[0])
km.plot_codo(max_clusters=10)

plt.sca(axes[1])
km.plot_silhouette(max_clusters=10)

plt.tight_layout()
plt.show()

## 5. Métricas del modelo

In [ ]:
import pandas as pd

print(f'Número de clusters : {km.n_clusters}')
print(f'Inercia            : {km.inercia:.2f}')
print(f'Silhouette         : {km.silhouette:.4f}')
print()
print('Distribución de observaciones por cluster:')
print(pd.Series(km.etiquetas).value_counts().sort_index())

> **Inercia:** suma de distancias cuadradas al centroide — menor es mejor.  
> **Silhouette ≥ 0.50** → clusters bien separados. **≥ 0.70** → excelente separación.

## 6. Visualizaciones

### 6.1 Mapa de calor — Perfil de clusters

In [ ]:
plt.figure(figsize=(16, 5))
km.plot_mapa_calor(titulo='Perfil de Clusters — Hotel Bookings (K-Means)')
plt.tight_layout()
plt.show()

### 6.2 Distribución de observaciones por cluster

In [ ]:
plt.figure(figsize=(8, 5))
km.plot_distribucion()
plt.tight_layout()
plt.show()

### 6.3 Diagramas de dispersión por cluster

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axes[0])
km.plot_dispersion('adr', 'lead_time', titulo='Tarifa Diaria vs Anticipación de Reserva')

plt.sca(axes[1])
km.plot_dispersion('adr', 'stays_in_week_nights', titulo='Tarifa Diaria vs Noches entre Semana')

plt.tight_layout()
plt.show()

### 6.4 Barras de centroides por cluster

In [ ]:
km.plot_barras(
    titulo='Perfiles de Clusters — Hotel Bookings (K-Means)',
    escala=True
)
plt.show()

### 6.5 Gráfico radar de clusters

In [ ]:
km.plot_radar(titulo='Radar de Clusters — Hotel Bookings (K-Means)')
plt.show()

## 7. Resumen estadístico por cluster

In [ ]:
km.resumen.style.background_gradient(cmap='Blues', axis=0).format('{:.2f}')

## 8. Conclusiones

- El **método del codo** y el **Silhouette** permiten elegir `k` de forma objetiva.
- El **mapa de calor** revela el perfil de cada cluster (e.g. reservas de lujo, cancelaciones frecuentes, etc.).
- K-Means es **no determinista**: distintas semillas pueden producir resultados diferentes. El parámetro `random_state=42` garantiza reproducibilidad.
- K-Means asume clusters **esféricos y de tamaño similar**. Si los clusters son irregulares, considera HAC con enlace completo o average.